# Limpeza de Áudio por Predição do LSTM

Objetivo: remover trechos preditos como ruido/silencio e exportar WAV limpo para o notebook de transcrição.

Observação importante: este notebook assume que o CSV de features e o WAV referem-se ao mesmo sinal temporal.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from chime_lstm_data import build_sequences
from lstm_model import LSTMClassifier, infer_records, load_model

In [ ]:
BASE_DIR = Path('.')
ARTIFACTS_DIR = BASE_DIR / 'results' / 'lstm_asr'
OUTPUT_DIR = BASE_DIR / 'results' / 'clean_audio'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ajuste para o arquivo que deseja limpar
CSV_PATH = BASE_DIR / 'chime6_kinect_dev.csv'
TARGET_SAMPLE_ID = 'S02_U06_block_0000'
WAV_PATH = BASE_DIR / 'audio' / 'dev' / 'S02_U06.CH1.wav'

MODEL_PATH = ARTIFACTS_DIR / 'lstm_asr_best.pt'
SCALER_PATH = ARTIFACTS_DIR / 'scaler_config.json'
BEST_PARAMS_PATH = ARTIFACTS_DIR / 'best_params.json'

THRESHOLD = 0.5
FADE_MS = 20.0

print('WAV alvo:', WAV_PATH)

## 1) Carregar artefatos do treino

In [ ]:
with open(SCALER_PATH, encoding='utf-8') as f:
    scaler_cfg = json.load(f)

with open(BEST_PARAMS_PATH, encoding='utf-8') as f:
    best_params = json.load(f)

feature_cols = scaler_cfg['feature_cols']
seq_len = int(scaler_cfg['seq_len'])
stride = int(best_params['stride'])

mean = np.asarray(scaler_cfg['mean'], dtype=np.float32)
scale = np.asarray(scaler_cfg['scale'], dtype=np.float32)

model = LSTMClassifier(
    input_dim=len(feature_cols),
    hidden_dim=int(best_params['hidden_dim']),
    num_layers=int(best_params['num_layers']),
    dropout_rate=float(best_params['dropout_rate']),
)
model = load_model(model, str(MODEL_PATH))
print('Modelo carregado.')

## 2) Carregar features do sample e inferir fala/ruído

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df['sample_id'] == TARGET_SAMPLE_ID].copy()
df = df.sort_values('timestamp_ms').reset_index(drop=True)

if df.empty:
    raise ValueError(f'Sample não encontrado: {TARGET_SAMPLE_ID}')

df.loc[:, feature_cols] = (df[feature_cols].values - mean) / (scale + 1e-8)

pack = build_sequences(
    df=df,
    feature_cols=feature_cols,
    seq_len=seq_len,
    stride=stride,
    label_mode='majority',
)

preds, scores = infer_records(model, pack.X, threshold=THRESHOLD)
print('records:', len(preds), 'speech_ratio:', float(preds.mean()) if len(preds) else 0.0)

## 3) Converter predição por record em máscara por frame

In [ ]:
n_frames = len(df)
vote_sum = np.zeros(n_frames, dtype=np.float32)
vote_cnt = np.zeros(n_frames, dtype=np.float32)

for i, p in enumerate(preds):
    start = i * stride
    end = start + seq_len
    if end > n_frames:
        break
    vote_sum[start:end] += float(p)
    vote_cnt[start:end] += 1.0

frame_mask = np.zeros(n_frames, dtype=np.float32)
valid = vote_cnt > 0
frame_mask[valid] = (vote_sum[valid] / vote_cnt[valid] >= 0.5).astype(np.float32)

# Frames sem voto (borda final) herdam o último estado conhecido
if np.any(valid):
    last_value = frame_mask[np.where(valid)[0][-1]]
    frame_mask[~valid] = last_value

print('frame speech ratio:', float(frame_mask.mean()))

## 4) Aplicar máscara no áudio e exportar

In [ ]:
audio, sr = sf.read(WAV_PATH)
if audio.ndim > 1:
    audio = audio.mean(axis=1)

timestamps_ms = df['timestamp_ms'].to_numpy(dtype=np.float64)

# Estima passo temporal por frame via mediana
if len(timestamps_ms) > 1:
    frame_step_ms = float(np.median(np.diff(timestamps_ms)))
else:
    frame_step_ms = 8.0

samples_per_frame = max(1, int(round((frame_step_ms / 1000.0) * sr)))
audio_mask = np.zeros(len(audio), dtype=np.float32)

for i, m in enumerate(frame_mask):
    a = i * samples_per_frame
    b = min(len(audio), (i + 1) * samples_per_frame)
    if a >= len(audio):
        break
    audio_mask[a:b] = m

# Crossfade curto para reduzir artefato de corte
fade_samples = max(1, int((FADE_MS / 1000.0) * sr))
if fade_samples > 1:
    kernel = np.ones(fade_samples, dtype=np.float32) / fade_samples
    audio_mask = np.convolve(audio_mask, kernel, mode='same')
    audio_mask = np.clip(audio_mask, 0.0, 1.0)

clean = audio * audio_mask

out_wav = OUTPUT_DIR / f'{TARGET_SAMPLE_ID}_clean.wav'
sf.write(out_wav, clean, sr)

coverage = float((audio_mask > 0.5).mean())
print('Áudio salvo em:', out_wav)
print('speech_coverage:', round(coverage, 4))